# 003-pose-search

# Imports

In [ ]:
import json
import os
import time

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.factory import build_features, correlator_config
from uavloc.geo import LatLon
from uavloc.matching.benchmark import run_benchmark, summarise
from uavloc.matching.correlator import PositionCorrelator
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "003-pose-search"
METHOD = "segmentation-labels"            # "vgg11" | "segmentation-labels"
MATCHING = "weighted-ncc"                 # "weighted-ncc" (class maps) | "cosine" (vgg11)
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water; weighted-ncc only
HEADING_SEARCH = True
SCALE_SEARCH = False
NOTE = ""
REGISTERED_RUN = ""                       # the proposal's run label; log_run numbers runs in the order they are logged
MACHINE = "Apple M1 Pro"                  # 001/002 were logged on the old Intel Mac
REPLAY_CHECKS = "artifacts/checks.json"   # same-machine replay of the baselines (checks.ipynb)
LOG_RUN = True                            # False: a dry run, nothing is appended to runs.jsonl
FRAME_LIMIT = None                        # None: all 44 frames; a small number only for a dry run

# the search schedule, fixed by the proposal
HEADING_RANGE_DEG = 36                    # coarse offsets from -36 to +36, added to the logged heading
HEADING_COARSE_STEP_DEG = 3
HEADING_FINE_SPAN_DEG = 3                 # fine offsets from -3 to +3 around the coarse best
HEADING_FINE_STEP_DEG = 1
SEARCH_SCALES = (0.5, 0.6, 0.7, 0.8, 0.9, 1.0)   # drone footprint over the footprint from the logged altitude
POSE_SCHEDULE = "greedy"                  # "greedy" (registered) | "joint": every offset x every scale (2026-10-20 amendment)
FINE_PASS_TOP_K = 1                       # "joint" only: fine pass around this many best coarse pairs (amendment 2: 3)

NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 15.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
TORCH_THREADS = 2                         # the CPU is shared with other runs

# 2025 correlator geometry, unchanged for every method
SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0

VGG11_CHECKPOINT = "../archive/uav_localization_2025/feature_models/models_finetrain/vgg11_drop_margin08_adamw_model_20250521_034900.pth"
DRONE_INPUT_PX = 448

SEG_CHECKPOINT_SAT = "pocs/village-seg-baseline/model-round3-bn-satellite.pt"
SEG_CHECKPOINT_DRONE = "pocs/village-seg-baseline/model-round3-bn-drone.pt"
SEG_CHANNELS = "all"                # "all" | "no-background"
DRONE_SEG_GSD_M = 0.10
MATCH_GSD_M = 0.385

LABEL_LOCATION = "gonchary-2025"               # hand-labelled satellite map, segmentation-labels only
LABEL_DATE = "ortho"
LABEL_DATASET = "ua-ortho-annotated"

FRAMES_DATASET = "drone-2025-easy"
SEG_SPLITS = "pocs/village-seg-baseline/results-round2.json"
SAT_IMAGE = "../archive/uav_localization_2025/data/satelites/satellite_gonchary.jpg"
SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(TORCH_THREADS)

In [ ]:
heading_offsets_deg = ([float(d) for d in range(-HEADING_RANGE_DEG, HEADING_RANGE_DEG + 1, HEADING_COARSE_STEP_DEG)]
                       if HEADING_SEARCH else [0.0])
heading_fine_offsets_deg = ([float(d) for d in range(-HEADING_FINE_SPAN_DEG, HEADING_FINE_SPAN_DEG + 1, HEADING_FINE_STEP_DEG)]
                            if HEADING_SEARCH else [0.0])
scales = [float(s) for s in SEARCH_SCALES] if SCALE_SEARCH else [1.0]
len(heading_offsets_deg), heading_fine_offsets_deg, scales

In [ ]:
RUN_CONFIG = {
    "method": METHOD, "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "heading_offsets_deg": heading_offsets_deg, "heading_fine_offsets_deg": heading_fine_offsets_deg, "scales": scales,
    "pose_schedule": POSE_SCHEDULE, "fine_pass_top_k": FINE_PASS_TOP_K,
    "vgg11_checkpoint": VGG11_CHECKPOINT, "drone_input_px": DRONE_INPUT_PX,
    "seg_checkpoint_sat": SEG_CHECKPOINT_SAT, "seg_checkpoint_drone": SEG_CHECKPOINT_DRONE,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": DRONE_SEG_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "label_location": LABEL_LOCATION, "label_date": LABEL_DATE, "label_dataset": LABEL_DATASET,
    "frames_dataset": FRAMES_DATASET, "frame_limit": FRAME_LIMIT, "sat_image": SAT_IMAGE,
    "registered_run": REGISTERED_RUN, "machine": MACHINE,
}

In [ ]:
replay = {row["baseline"]: row for row in json.loads((experiment_dir(EXP, create=False) / REPLAY_CHECKS).read_text())["full_replay"]}
same_machine_replay = {f"replay_{run_id.replace('.', '_')}_this_machine_m": row["this_machine_mean_error_m"] for run_id, row in replay.items()}
same_machine_replay

In [ ]:
frames = load_frames(FRAMES_DATASET)[:FRAME_LIMIT]
len(frames)

In [ ]:
satellite = GeoImage.load(REPO_ROOT / SAT_IMAGE, LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
satellite.mpp

In [ ]:
def frame_subsets(frames: list[DroneFrame], seg_splits_path: str) -> dict[str, set[str]]:
    splits = json.loads((REPO_ROOT / seg_splits_path).read_text())["splits"]
    seg_seen = set(splits["drone_train"] + splits["drone_dropped"])
    return {
        "all": {f.name for f in frames},
        "2025_train": {f.name for f in frames if "images_2_v3_renamed_small_easy" in f.picked_from},
        "2025_val": {f.name for f in frames if "images_v6_renamed_easy" in f.picked_from},
        "seg_heldout": {f.name for f in frames if f.name not in seg_seen},
    }

In [ ]:
subsets = frame_subsets(frames, SEG_SPLITS)
{name: len(members) for name, members in subsets.items()}

# Model

In [ ]:
features = build_features(RUN_CONFIG, satellite, DEVICE)
correlator = PositionCorrelator(features, correlator_config(RUN_CONFIG))
correlator.config

# Run

In [ ]:
start = time.perf_counter()
records = pd.concat([run_benchmark(correlator, frames, noise_m, N_DIRECTIONS, SEED, progress=True) for noise_m in NOISE_M],
                    ignore_index=True)
wall_time_s = time.perf_counter() - start
records.groupby("noise_m")[["error_m", "reachable", "heading_offset_deg", "scale", "pose_hypotheses"]].mean()

In [ ]:
def pose_stats(records: pd.DataFrame, noise_m: float) -> dict[str, float]:
    """The pose the search chose at one noise level: how far it moved from the log."""
    level = records[(records["noise_m"] == noise_m) & (records["status"] == "ok")]
    return {
        "median_abs_heading_offset_deg": float(level["heading_offset_deg"].abs().median()),
        "share_heading_moved": float((level["heading_offset_deg"] != 0).mean()),
        "median_scale": float(level["scale"].median()),
        "share_scale_below_1": float((level["scale"] < 1).mean()),
        "mean_pose_hypotheses": float(level["pose_hypotheses"].mean()),
    }

In [ ]:
metrics = {**summarise(records, subsets, PRIMARY_NOISE_M), **pose_stats(records, PRIMARY_NOISE_M), "wall_time_s": wall_time_s,
           **same_machine_replay}
{k: v for k, v in metrics.items() if "_at_" not in k}

# Plots

In [ ]:
title = f"{METHOD}, {MATCHING}, heading search {HEADING_SEARCH}, scale search {SCALE_SEARCH}, {POSE_SCHEDULE} top {FINE_PASS_TOP_K}"

In [ ]:
fig_cdf, ax = plt.subplots(figsize=(6, 4))
for noise_m, group in records.groupby("noise_m"):
    errors = np.sort(group["error_m"].to_numpy())
    line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
    ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
ax.set_title(title, fontsize=9); ax.legend(); ax.grid(alpha=.3)
fig_cdf.tight_layout()

In [ ]:
primary = records[records["noise_m"] == PRIMARY_NOISE_M]
subset_means = {name: primary[primary["frame"].isin(members)]["error_m"].mean() for name, members in subsets.items()}
fig_subsets, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(subset_means), list(subset_means.values()))
ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1, label="no correction")
ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title(title, fontsize=9); ax.legend()
fig_subsets.tight_layout()

In [ ]:
fig_reach, ax = plt.subplots(figsize=(6, 3.5))
for reachable, group in primary.groupby("reachable"):
    ax.hist(group["error_m"], bins=30, range=(0, 40), alpha=.6, label=f"truth in window: {reachable}")
ax.axvline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
ax.set_xlabel(f"error @ {PRIMARY_NOISE_M:g} m injected, m"); ax.set_ylabel("draws"); ax.set_title(title, fontsize=9); ax.legend()
fig_reach.tight_layout()

In [ ]:
fig_pose, axes = plt.subplots(1, 2, figsize=(10, 3.5))
offset_edges = np.arange(-HEADING_RANGE_DEG - HEADING_FINE_SPAN_DEG - 0.5, HEADING_RANGE_DEG + HEADING_FINE_SPAN_DEG + 1.5)
axes[0].hist(primary["heading_offset_deg"], bins=offset_edges)
axes[0].set_xlabel("chosen heading offset, deg"); axes[0].set_ylabel("draws")
scale_values, scale_counts = np.unique(primary["scale"], return_counts=True)
axes[1].bar([f"{s:g}" for s in scale_values], scale_counts)
axes[1].set_xlabel("chosen footprint scale"); axes[1].set_ylabel("draws")
fig_pose.suptitle(f"{title}, @ {PRIMARY_NOISE_M:g} m", fontsize=9)
fig_pose.tight_layout()

In [ ]:
run_id = "dry"
if LOG_RUN:
    run = log_run(
        exp=EXP,
        config=RUN_CONFIG,
        metrics=metrics,
        note=NOTE,
        figures=[fig_cdf, fig_subsets, fig_reach, fig_pose],
        seed=SEED,
        device=DEVICE,
    )
    run_id = run["run"]
run_id

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
records.to_csv(artifacts / f"records-{run_id}.csv", index=False)